In [12]:
import os
import argparse
import builtins
import time
import numpy as np

import torch
import torch.nn.functional as F
from torch import multiprocessing as mp
import torch.distributed as dist

import utils
from model import AVENet
from DatasetLoader import GetAudioVideoDataset
from train import train, validate
# from datasets import get_train_dataset, get_test_dataset


import argparse

In [9]:
class Args:
    def __init__(self):
        # Model and experiment configurations
        self.model_dir = './checkpoints'  # Directory to save model checkpoints
        self.experiment_name = 'hdvsl_vggss'  # Experiment name for checkpointing and logging

        # Dataset paths and configurations
        self.trainset = 'vggss'  # Name of the training dataset (e.g., flickr, vggss)
        self.testset = 'vggss'  # Name of the test dataset (e.g., flickr, vggss)
        self.image_size = 224  # Height and width of inputs

        # Model hyperparameters as used in AVENet class
        self.epsilon = 0.65  # Threshold for positive cases in similarity calculation
        self.epsilon2 = 0.4  # Threshold for negative cases in similarity calculation
        self.tri_map = True  # Use tri-map for additional negative cases
        self.Neg = True  # Include negative samples in similarity calculation
        self.random_threshold = 0.03  # Threshold for random sampling (if used)

        # Training parameters
        self.epochs = 20  # Total number of training epochs
        self.batch_size = 128  # Batch size for training
        self.init_lr = 0.0001  # Initial learning rate
        self.seed = 12345  # Random seed for reproducibility

        # Distributed training parameters
        self.workers = 8  # Number of workers for data loading
        self.gpu = None  # GPU id to use
        self.world_size = 1  # Total number of nodes for distributed training
        self.rank = 0  # Node rank for distributed training
        self.node = 'localhost'  # Node hostname
        self.port = 12345  # Port for distributed training communication
        self.dist_url = 'tcp://localhost:12345'  # URL for distributed training
        self.multiprocessing_distributed = False  # Use multi-processing distributed training

args = Args()

In [ ]:
# 모델 초기화
model = AVENet(args)
model.cuda()  # GPU 설정
optimizer, scheduler = utils.build_optimizer_and_scheduler_adam(model, args)  # 옵티마이저 설정

# 데이터 로더 설정
train_dataset = GetAudioVideoDataset(args, mode='train')
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=args.batch_size, shuffle=False, num_workers=16)
test_dataset = GetAudioVideoDataset(args, mode='test')
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, num_workers=16)

# 학습 및 검증 루프 실행
for epoch in range(args.epochs):
    train(train_loader, model, optimizer, epoch, args)
    validate(test_loader, model, args)

ValueError: too many values to unpack (expected 4)

: 